# 淘宝用户购物行为
## 概述
本数据集包含了2017年11月25日至2017年12月3日之间，有行为的约一百万随机用户的所有行为（行为包括点击、购买、加购、喜欢）。数据集的组织形式和MovieLens-20M类似，即数据集的每一行表示一条用户行为，由用户ID、商品ID、商品类目ID、行为类型和时间戳组成，并以逗号分隔。
## 介绍
用户ID	整数类型，序列化后的用户ID<br />
商品ID	整数类型，序列化后的商品ID<br />
商品类目ID	整数类型，序列化后的商品所属类目ID<br />
行为类型	字符串，枚举类型，包括('pv', 'buy', 'cart', 'fav')<br />
时间戳	行为发生的时间戳<br />
<br />
pv	商品详情页pv，等价于点击<br />
buy	商品购买<br />
cart 将商品加入购物车<br />
fav	收藏商品<br />
## 数据量
维度	数量<br />
用户数量	987,994<br />
商品数量	4,162,024<br />
用户数量	987,994<br />
商品类目数量	9,439<br />
所有行为数量	100,150,807<br />
## 引用
- 1.Han Z, Xiang L, Pengye Z, et al. 2018. Learning Tree-based Deep Model for Recommender Systems. In Proceedings of the 24th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining.
- 2.Han Z, Daqing C, Ziru X, et al. 2019. Joint Optimization of Tree-based Index and Deep Model for Recommender Systems. In Advances in Neural Information Processing Systems.
- 3.Jingwei Z, Ziru X, Wei D, et al. 2020. Learning Optimal Tree Models under Beam Search. In International Conference on Machine Learning.

---

# 解决问题
将商品依照热度和转化诊断模型进行分类，寻找明星商品、潜力商品、问题商品。<br />
将用户依照AIDA矩阵进行分类，寻找高价值客户。通过BG/NBD模型判断各类型用户总价值。<br />
根据数据推理出商品推荐模型，优化用户的转化路径，促进用户在平台完成更多消费。<br />

---

# 初始化与数据导入

In [33]:
#导入库和文件
import sqlite3 as sql3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import duckdb
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

file_name = 'UserBehavior.csv'
db = 'UserBehavior.db'

In [20]:
#删除可能存在的已有表
_ = duckdb.connect(db)

_.execute("""
    DROP TABLE Raw;
    DROP TABLE Cleaned;
""")

_.close()

In [21]:
#创建数据库
_ = duckdb.connect(db)

_.execute("""
    CREATE TABLE Raw AS 
    SELECT 
        column0 AS user_id,
        column1 AS good_id,
        column2 AS prop_id,
        column3 AS act,
        column4 AS timestamp
    FROM read_csv('UserBehavior.csv');
""")

_.close()
print('数据库创建成功！')

数据库创建成功！


In [22]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Raw LIMIT 5;").fetchdf())
#print(_.execute('SUMMARIZE Raw;').fetchdf())
_.close()

   user_id  good_id  prop_id act   timestamp
0        1  2268318  2520377  pv  1511544070
1        1  2333346  2520771  pv  1511561733
2        1  2576651   149192  pv  1511572885
3        1  3830808  4181361  pv  1511593493
4        1  4365585  2520377  pv  1511596146


---

# 数据清洗

因源文件数据量过大，将database数据导入到dataframe再使用pandas进行数据清洗效率将十分低下，故选择直接在database中进行数据清洗。

In [23]:
_ = duckdb.connect(db)

#清除重复值
_.execute("""
    CREATE TABLE Cleaned AS 
    SELECT DISTINCT * FROM Raw
""")

#清除缺失值
_.execute("""
    DELETE FROM Cleaned
    WHERE user_id IS NULL 
    OR good_id IS NULL 
    OR act IS NULL
    OR timestamp IS NULL
    OR prop_id IS NULL;
""")

#清除逻辑异常值
_.execute("""
    DELETE FROM Cleaned
    WHERE act NOT IN ('pv','cart','fav','buy')
    OR (timestamp < 1511539200 OR timestamp > 1512230400);
""")

print('数据清洗完成！')
_.close()

数据清洗完成！


In [24]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Cleaned LIMIT 5;").fetchdf())
#print(_.execute('SUMMARIZE Cleaned;').fetchdf())
_.close()

   user_id  good_id  prop_id act   timestamp
0   127982  1405406  1349561  pv  1512122561
1   127996   654630  4015820  pv  1511696971
2   128000  1098114  3776866  pv  1511586855
3   128000  4705766  3607361  pv  1511933898
4   128000   599068   982926  pv  1511938014


---

# 数据格式化

---

# 用户转化漏斗分析

按照商品汇总其pv,cart,fav,buy的数量创建两个新的table，分别按照商品id和商品类目id汇总分析。<br/>
定义一个新的数据为like，其值为cart和fav的加和。<br/>
第一个表的列名称分别为商品id，pv2like，like2buy，pv2buy。<br/>
第二个表的列名称分别为商品类目id，pv2like，like2buy，pv2buy。<br/>
清除掉部分访问极少的商品(pv<1000)以保证转化率的真实性。<br/>

In [29]:
#删除可能存在的已有表
_ =duckdb.connect(db)
_.execute("""
    DROP TABLE Good_Funnel_Analysis;
    DROP TABLE Prop_Funnel_Analysis;
""")
_.close()

In [30]:
#构建表
_ =duckdb.connect(db)
_.execute("""
    CREATE TABLE Good_Funnel_Analysis AS 
        WITH metrics AS (
            SELECT 
                good_id,
                COUNT(CASE WHEN act = 'pv' THEN 1 END) AS pv_cnt,
                COUNT(CASE WHEN act IN ('fav','cart') THEN 1 END) AS like_cnt,
                COUNT(CASE WHEN act = 'buy' THEN 1 END) AS buy_cnt
            FROM Cleaned
            GROUP BY good_id
            HAVING COUNT(CASE WHEN act = 'pv' THEN 1 END) > 100
        )
        SELECT 
            good_id,
            ROUND(like_cnt * 100.0 / NULLIF(pv_cnt, 0), 4) AS pv2like_rate,
            ROUND(buy_cnt * 100.0 / NULLIF(like_cnt, 0), 4) AS like2buy_rate,
            ROUND(buy_cnt * 100.0 / NULLIF(pv_cnt, 0), 4) AS pv2buy_rate
        FROM metrics;  
""")
_.execute("""
    CREATE TABLE Prop_Funnel_Analysis AS
        WITH metrics AS (
                SELECT 
                    prop_id,
                    COUNT(CASE WHEN act = 'pv' THEN 1 END) AS pv_cnt,
                    COUNT(CASE WHEN act IN ('fav','cart') THEN 1 END) AS like_cnt,
                    COUNT(CASE WHEN act = 'buy' THEN 1 END) AS buy_cnt
                FROM Cleaned
                GROUP BY prop_id
                HAVING COUNT(CASE WHEN act = 'pv' THEN 1 END) > 100
            )
            SELECT 
                prop_id,
                ROUND(like_cnt * 100.0 / NULLIF(pv_cnt, 0), 4) AS pv2like_rate,
                ROUND(buy_cnt * 100.0 / NULLIF(like_cnt, 0), 4) AS like2buy_rate,
                ROUND(buy_cnt * 100.0 / NULLIF(pv_cnt, 0), 4) AS pv2buy_rate
            FROM metrics;  
""")
_.close()

In [31]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Good_Funnel_Analysis LIMIT 5;").fetchdf())
print(_.execute("SELECT * FROM Prop_Funnel_Analysis LIMIT 5;").fetchdf())
_.close()

   good_id  pv2like_rate  like2buy_rate  pv2buy_rate
0   698543        7.2628        56.9364       4.1352
1  3503843        7.8947         0.0000       0.0000
2  1932379        8.9109        66.6667       5.9406
3  2439074       13.6496         4.9645       0.6776
4  1175063        8.7500        92.8571       8.1250
   prop_id  pv2like_rate  like2buy_rate  pv2buy_rate
0  3549297        7.6772        21.6837       1.6647
1  2096639        3.5309        22.6876       0.8011
2  4969568       14.7072        46.9516       6.9052
3  2920476        7.6702         5.8185       0.4463
4  1526484       14.0479        13.5440       1.9026


In [ ]:
#商品分类

# 用户价值与行为特征分析

# 商品价值分析 

# 转化率预测

# 用户行为路径预测